# 03. Exploratory Data Analysis (EDA)
Comprehensive analysis of revenue growth, category performance, seller concentration, regional delivery SLAs, payment preferences, and the customer satisfaction penalty of late deliveries.

In [1]:
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
import pandas as pd
import numpy as np

master = pd.read_csv("data/processed/olist_orders_master.csv")
delivered = pd.read_csv("data/processed/olist_delivered_orders.csv")
orders_items = master[master["item_count"].notna()].copy()

print("=== Top 10 Categories by Item Revenue ===")
cat_summary = (
    orders_items.groupby("primary_category", as_index=False)
    .agg(orders=("order_id", "count"), revenue=("total_price", "sum"), avg_price=("total_price", "mean"))
    .sort_values("revenue", ascending=False)
    .head(10)
)
print(cat_summary.round(2).to_string(index=False))

print("\n=== Top 10 Customer States by Delivered Volume & SLA ===")
state_sla = (
    delivered.groupby("customer_state", as_index=False)
    .agg(
        delivered_orders=("order_id", "count"),
        avg_delivery_days=("actual_delivery_days", "mean"),
        late_rate_pct=("is_late", lambda s: round(s.mean() * 100, 2)),
        avg_review_score=("review_score", "mean")
    )
    .sort_values("delivered_orders", ascending=False)
    .head(10)
)
print(state_sla.round(2).to_string(index=False))

print("\n=== Customer Review Score by Delivery SLA Status ===")
delivered["sla_group"] = np.where(delivered["is_late"] == 1, "Late Delivery", "On-Time / Early")
print(delivered.groupby("sla_group").agg(
    orders=("order_id", "count"),
    mean_review_score=("review_score", "mean"),
    median_review_score=("review_score", "median")
).round(2).to_string())

=== Top 10 Categories by Item Revenue ===
     primary_category  orders    revenue  avg_price
        health_beauty    8801 1259312.82     143.09
        watches_gifts    5607 1205980.88     215.08
       bed_bath_table    9328 1037777.68     111.25
       sports_leisure    7683  987722.31     128.56
computers_accessories    6670  911673.70     136.68
      furniture_decor    6330  727567.60     114.94
           cool_stuff    3603  635460.18     176.37
           housewares    5821  631945.90     108.56
                 auto    3880  593117.61     152.87
                 toys    3870  484694.46     125.24

=== Top 10 Customer States by Delivered Volume & SLA ===
customer_state  delivered_orders  avg_delivery_days  late_rate_pct  avg_review_score
            SP             40489               8.76           5.90              4.25
            RJ             12348              15.31          13.48              3.97
            MG             11351              12.01           5.61       